# Presenter walkthrough — `session2_part2_gpu.ipynb`

**Solved copy, for driving the session live.** It clones the `session2-part2-solution` branch, so every cell below runs as-is with no editing; attendees run the exercise notebook, which does not. The narration lives in the deck, `presentation/session2_part2.pdf`.

Generated by `presentation/generate_walkthrough.py` — do not edit this notebook; re-run the script.


# SC26 Tutorial — Session 2, Part 2: distributing Linalg on the GPU

Part 1 tiled Linalg ops into blocks and ran them faster on a single CPU thread. Part 2 uses the same transformation to **distribute** them across a GPU:

1. **The GPU payloads** — part 1's `linalg.matvec` and `linalg.matmul`, with their buffers registered for the GPU.
2. **A schedule that distributes** — `tile_using_forall` turns loops into `scf.forall`, whose iterations may run in parallel, and says which GPU ids each one maps onto: an outer one onto GPU **blocks**, a nested one onto the **threads** in each block. The schedule is written for matvec, and you modify it for matmul.

The schedule itself builds the `gpu.launch`. Each version is then compiled for this GPU, JIT-executed and checked against numpy.

**This part needs a GPU runtime: Runtime -> Change runtime type -> T4 GPU.**

**You write MLIR in a source file; every notebook cell is provided.**

## Setup

**Run the cells in order, starting here.** Later cells use names defined by earlier ones, so skipping ahead fails with `NameError`. `Ctrl+F8` (run every cell above) catches you up if you lose your place.

Same prebuilt MLIR toolchain as part 1.

In [ ]:
REPO = "MaheshRavishankar/sc26_tutorial"
RELEASE_TAG = "session1-toolchain-v1"
ASSET_NAME = "mlir-tutorial-llvm.tar.gz"

In [ ]:
import os
import requests

# Set MLIR_TUTORIAL_INSTALL_DIR to iterate locally (e.g. in VS Code) against
# an already-built toolchain -- skips the download entirely if it's already
# there. Unset in Colab, so this always defaults to the normal /content path.
INSTALL_DIR = os.environ.get("MLIR_TUTORIAL_INSTALL_DIR", "/content/mlir-tutorial-llvm")

if os.path.exists(f"{INSTALL_DIR}/bin/mlir-opt"):
    print(f"Using existing toolchain at {INSTALL_DIR}")
else:
    tar_path = f"{INSTALL_DIR}.tar.gz"
    url = f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(tar_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

    !mkdir -p {INSTALL_DIR}
    !tar xf {tar_path} -C {INSTALL_DIR} --strip-components=1

In [ ]:
import os, sys

os.environ["PATH"] = f"{INSTALL_DIR}/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{INSTALL_DIR}/python_packages/mlir_core")

!mlir-opt --version

In [ ]:
from mlir.ir import Context, Module
from mlir.passmanager import PassManager
from mlir.execution_engine import ExecutionEngine
import mlir.runtime as rt
import ctypes
import numpy as np

print("MLIR Python bindings imported OK")

Also clone this repo -- every `.mlir` source below lives here.

In [ ]:
# Set MLIR_TUTORIAL_REPO_DIR to iterate locally against an already-cloned
# checkout -- skips the clone entirely if it's already there.
REPO_DIR = os.environ.get("MLIR_TUTORIAL_REPO_DIR", "/content/sc26_tutorial")
if os.path.isdir(REPO_DIR):
    print(f"Using existing repo checkout at {REPO_DIR}")
else:
    # --no-single-branch: every branch's tip, still at depth 1. It costs
    # nothing measurable, and it means the solution branches are already in
    # your checkout if you want a solved file out of one.
    WALKTHROUGH_BRANCH = "session2-part2-solution"
    !git clone --depth 1 --no-single-branch --branch {WALKTHROUGH_BRANCH} https://github.com/{REPO} {REPO_DIR}

Find the GPU. Its compute capability (`sm_75` on a T4) is the target the kernel gets compiled for.

In [ ]:
import shutil
import subprocess

assert shutil.which("nvidia-smi"), (
    "no GPU found -- switch the runtime: Runtime -> Change runtime type -> T4 GPU")


def gpu_chip_and_features():
    """Detect the GPU's compute capability and pick a PTX ISA version new
    enough to support it."""
    cc = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc.split(".")
    return f"sm_{major}{minor}", "+ptx78"


GPU_CHIP, GPU_PTX_FEATURES = gpu_chip_and_features()
print("GPU target:", GPU_CHIP, GPU_PTX_FEATURES)

## Step 1 — the GPU payloads

Two files, one op each, both complete:

- **`session2/linalg_matvec_gpu.mlir`** — `@matvec`;
- **`session2/linalg_matmul_gpu.mlir`** — `@matmul`.

They are part 1's files with one addition. A GPU kernel reads and writes the host's buffers directly, so each buffer is registered with the CUDA driver, with `gpu.host_register`, before the op runs. The `memref.cast` to an unranked memref is there only because `gpu.host_register` takes one.

In [ ]:
# Opens the files. On Colab, files.view() puts them in the editor pane.
# Running locally there is no google.colab, so this prints the paths instead
# and you open them in your own editor.
MATVEC_FILE = f"{REPO_DIR}/session2/linalg_matvec_gpu.mlir"
MATMUL_FILE = f"{REPO_DIR}/session2/linalg_matmul_gpu.mlir"

try:
    from google.colab import files

    files.view(MATVEC_FILE)
    files.view(MATMUL_FILE)
except ModuleNotFoundError:
    print(f"open in your editor:\n    {MATVEC_FILE}\n    {MATMUL_FILE}")

Round-trip both through `mlir-opt`.

In [ ]:
!mlir-opt {MATVEC_FILE}
assert _exit_code == 0
!mlir-opt {MATMUL_FILE}
assert _exit_code == 0

## Step 2 — a schedule that distributes

**`session2/distribute_linalg.mlir`** is written for matvec. Like part 1's schedule it matches any Linalg op by interface. It tiles twice with `tile_using_forall`, which produces `scf.forall` loops, and gives each loop a **mapping**: which GPU id it becomes.

1. `tile_sizes [32]`, mapped to `#gpu.block<x>` — an outer `scf.forall` over blocks of 32 rows, 4 of them for matvec's 128 rows. Each iteration becomes a GPU block, indexed by its block id in x.
2. `tile_sizes [1]`, mapped to `#gpu.thread<x>`, on the op inside each block — a nested `scf.forall` over that block's 32 rows, one row per iteration. Each iteration becomes a thread, indexed by its thread id in x.

Two more transform ops then build the kernel from those mappings: `transform.gpu.map_forall_to_blocks` puts the outer loop into a `gpu.launch` with one block per iteration, and `transform.gpu.map_nested_forall_to_threads` turns the nested loop into the threads of each block, `block_dims = [32, 1, 1]` of them. For matvec that is 4 blocks of 32 threads, each thread computing one `out[i]`.

Applied to matmul, the same schedule distributes along rows only. Matmul has 128 rows, so that is 4 blocks of 32 threads, but each thread computes a whole row of `c`, running over `j` and `k` itself.

The cells below apply it to matvec, then to matmul.

In [ ]:
# Opens the file you edit. On Colab, files.view() puts it in the editor pane.
# Running locally there is no google.colab, so this prints the path instead.
SCHEDULE = f"{REPO_DIR}/session2/distribute_linalg.mlir"

try:
    from google.colab import files

    files.view(SCHEDULE)
except ModuleNotFoundError:
    print(f"open in your editor -- you edit this:\n    {SCHEDULE}")

Round-trip the schedule through `mlir-opt`.

In [ ]:
!mlir-opt {SCHEDULE}
assert _exit_code == 0

### Matvec

Apply the schedule to matvec. The result is a `gpu.launch`. On its first line, `blocks(...) in (...)` sizes the grid, 4 blocks in x, and `threads(...) in (...)` sizes each block, 32 threads in x.

The body is what one thread runs. `gpu.block_id x` picks the block's 32-row slice of `mat` and `out`, `gpu.thread_id x` picks the thread's row within it, and a `linalg.matvec` works on that one row. Mapping only restructures: the op inside is still Linalg.

In [ ]:
MATVEC_DISTRIBUTED = "linalg_matvec_gpu_distributed.mlir"

!mlir-opt {MATVEC_FILE} \
    --transform-preload-library=transform-library-paths={SCHEDULE} \
    --transform-interpreter \
    -o {MATVEC_DISTRIBUTED} 2> schedule_warnings.txt
assert _exit_code == 0

# The interpreter reports a schedule that would let GPU threads race on the
# same output element as a warning, and emits the racy loops anyway.
schedule_warnings = open("schedule_warnings.txt").read()
print(schedule_warnings, end="")
assert "not thread safe" not in schedule_warnings, (
    "This schedule distributes a reduction loop of @matvec, so GPU threads would "
    "race to update the same output element.")

!cat {MATVEC_DISTRIBUTED}

Compiling and running it needs three things part 1's CPU runs did not:

- **`convert-linalg-to-loops`** first, as in part 1: it expands the `linalg.matvec` left inside the launch into a loop over the thread's one row and the reduction over `j`;
- **`gpu-lower-to-nvvm-pipeline`** compiles the `gpu.launch` body into a real GPU binary for this GPU's compute capability and rewrites the host side to launch it;
- **`shared_libs`** pointing at the CUDA runtime wrapper library, and **`engine.initialize()`**, which runs the global constructor that loads that compiled binary. Without it every CUDA call fails with `CUDA_ERROR_INVALID_HANDLE`.

The buffers stay ordinary numpy arrays; the `gpu.host_register` in the payload is what lets the kernel reach them.

In [ ]:
GPU_LOWERING_PIPELINE = (
    "builtin.module("
    "func.func(convert-linalg-to-loops),"
    "func.func(llvm-request-c-wrappers),"
    "expand-strided-metadata,"
    "lower-affine,"
    f"gpu-lower-to-nvvm-pipeline{{cubin-chip={GPU_CHIP} cubin-format=bin cubin-features={GPU_PTX_FEATURES}}}"
    ")"
)

GPU_SHARED_LIBS = [
    f"{INSTALL_DIR}/lib/libmlir_cuda_runtime.so",
    f"{INSTALL_DIR}/lib/libmlir_runner_utils.so",
    f"{INSTALL_DIR}/lib/libmlir_c_runner_utils.so",
]

REGISTERED = []


def memref_args(*arrays):
    return [ctypes.pointer(ctypes.pointer(rt.get_ranked_memref_descriptor(x))) for x in arrays]


def run_on_gpu(path, entry_point, args, expected):
    """Lower `path` to LLVM plus a compiled GPU kernel, JIT it, and call
    `entry_point` on fresh copies of `args`. The last one is the output, which
    the op accumulates into, so it starts at zero."""
    ctx = Context()
    module = Module.parseFile(path, ctx)
    PassManager.parse(GPU_LOWERING_PIPELINE, ctx).run(module.operation)
    engine = ExecutionEngine(module, opt_level=3, shared_libs=GPU_SHARED_LIBS)
    engine.initialize()

    args = [x.copy() for x in args]
    REGISTERED.extend(args)
    args[-1][...] = 0
    engine.invoke(entry_point, *memref_args(*args))
    err = float(np.max(np.abs(args[-1] - expected)))
    print(f"{entry_point}: result {args[-1].shape}, max abs error vs numpy = {err:.3g}")
    np.testing.assert_allclose(args[-1], expected, rtol=1e-3, atol=1e-3)
    print("OK")

# The matvec function takes a 128 x 512 matrix and a 512-vector, and writes
# 128 outputs.
ROWS, COLS = 128, 512
rng = np.random.default_rng(0)
mat = rng.random((ROWS, COLS), dtype=np.float32)
vec = rng.random(COLS, dtype=np.float32)
out = np.zeros(ROWS, dtype=np.float32)

run_on_gpu(MATVEC_DISTRIBUTED, "matvec", [mat, vec, out], mat @ vec)

### Matmul, with the schedule as provided

The same schedule, unchanged, on matmul. Matmul has 128 rows, so it is 4 blocks of 32 threads, but look at the `linalg.matmul` inside the `gpu.launch`: it writes a 1 x 256 slice of `c`. Each thread computes a whole row of `c`, running over all 256 columns `j` and all 512 `k` itself.

In [ ]:
MATMUL_ROWS = "linalg_matmul_gpu_rows.mlir"

!mlir-opt {MATMUL_FILE} \
    --transform-preload-library=transform-library-paths={SCHEDULE} \
    --transform-interpreter \
    -o {MATMUL_ROWS} 2> schedule_warnings.txt
assert _exit_code == 0

# The interpreter reports a schedule that would let GPU threads race on the
# same output element as a warning, and emits the racy loops anyway.
schedule_warnings = open("schedule_warnings.txt").read()
print(schedule_warnings, end="")
assert "not thread safe" not in schedule_warnings, (
    "This schedule distributes a reduction loop of @matmul, so GPU threads would "
    "race to update the same output element.")

!cat {MATMUL_ROWS}

In [ ]:
M, N, K = 128, 256, 512  # c is M x N; a is M x K; b is K x N
a = rng.random((M, K), dtype=np.float32)
b = rng.random((K, N), dtype=np.float32)
c = np.zeros((M, N), dtype=np.float32)

run_on_gpu(MATMUL_ROWS, "matmul", [a, b, c], a @ b)

### Matmul, distributed along columns too

**Exercise: modify `distribute_linalg.mlir` for matmul, so that it distributes along columns too — blocks of 32 by 32 threads, with each thread computing one output element `c[i, j]`. Map the columns (`j`) to x and the rows (`i`) to y, for both the blocks and the threads.**

After every edit, round-trip the schedule first, with the next cell. It checks that the schedule is well-formed, not that it does what you meant. Then run the three cells after it, in order: they apply the schedule to matmul, check the IR it produces, and run it on the GPU.

In [ ]:
!mlir-opt {SCHEDULE}
assert _exit_code == 0

Apply the schedule to matmul.

In [ ]:
MATMUL_DISTRIBUTED = "linalg_matmul_gpu_distributed.mlir"

!mlir-opt {MATMUL_FILE} \
    --transform-preload-library=transform-library-paths={SCHEDULE} \
    --transform-interpreter \
    -o {MATMUL_DISTRIBUTED} 2> schedule_warnings.txt
assert _exit_code == 0

# The interpreter reports a schedule that would let GPU threads race on the
# same output element as a warning, and emits the racy loops anyway.
schedule_warnings = open("schedule_warnings.txt").read()
print(schedule_warnings, end="")
assert "not thread safe" not in schedule_warnings, (
    "This schedule distributes a reduction loop of @matmul, so GPU threads would "
    "race to update the same output element.")

!cat {MATMUL_DISTRIBUTED}

**This is the check on the exercise.** It checks two things in the IR above. Distributed along columns as well as rows, each thread computes one output element, so the output operand of the innermost `linalg.matmul` is a 1 by 1 slice of `c`. And with the columns on x, that slice is cut at `[%thread_id_y, %thread_id_x]`: the thread id in y picks the row, the one in x the column.

In [ ]:
import re

matmul_line = next(line for line in open(MATMUL_DISTRIBUTED) if "linalg.matmul" in line)
shapes = re.findall(r"memref<(\d+x\d+)xf32", matmul_line)
print("shapes the innermost linalg.matmul works on (a, b, c):", shapes)
assert shapes and shapes[-1] == "1x1", (
    "matmul is not distributed along columns yet -- each thread still computes "
    f"a {shapes[-1] if shapes else '?'} slice of c rather than one element")

out_operand = re.search(r"outs\((%\w+)", matmul_line).group(1)
c_slice = next(line for line in open(MATMUL_DISTRIBUTED)
               if line.strip().startswith(f"{out_operand} = memref.subview"))
indices = re.search(r"memref\.subview %\w+\[([^\]]*)\]", c_slice).group(1)
print(f"c's 1x1 slice is cut at [{indices}]")
assert indices.replace(" ", "") == "%thread_id_y,%thread_id_x", (
    "each thread computes one element, but x is not on the columns yet -- map the "
    "column loop to #gpu.block<x> and #gpu.thread<x>, and the row loop to y")

Run it on the GPU. In the `gpu.launch` above, `blocks(...) in (...)` is 8 by 4: x counts the 8 blocks of columns, y the 4 blocks of rows. Each block is 32 by 32 threads, from `block_dims`. Lowered, the `linalg.matmul` on one row of `a` and one column of `b` leaves only the 512 iterations of `k`.

In [ ]:
run_on_gpu(MATMUL_DISTRIBUTED, "matmul", [a, b, c], a @ b)

### Why x on the columns?

The exercise asked for x on the columns, and the check enforced it. The numpy check would not have: the result is the same with x on the rows. What changes is how the threads touch memory.

`c` is row-major, so `c[i, j]` and `c[i, j+1]` are adjacent. A warp is 32 threads with consecutive `threadIdx.x` and the same `threadIdx.y`. With x on `j`, a warp's 32 threads write `c[i, j]`, `c[i, j+1]`, ... `c[i, j+31]`: 32 adjacent floats, which the GPU **coalesces** into one memory transaction. They also read 32 adjacent elements of a row of `b`, and all read the same `a[i, k]`.

With x on `i` instead, a warp would write `c[i, j]`, `c[i+1, j]`, ... `c[i+31, j]`: 32 floats a row apart, 1 KB at this size, one transaction each. Session 1 part 2's hand-written kernel made the same choice as this schedule: x on the columns.

The mapping is what decides it, and in the schedule it is one attribute per loop.